# Data Preprocessing
In this notebook, we will prepare our dataset for machine learning. Data preprocessing involves handling missing values, treating outliers, scaling features, and splitting the dataset into training and testing sets.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../dataset/Final_Flood_Project_Dataset.csv')
df.head()

## 2. Missing Value Handling
First, we check for missing values. If there are any, we impute them. For numerical data, replacing missing values with the median is generally robust against outliers.

In [ ]:
# Check missing values
print("Missing values before imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Impute missing values with the median for each column
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
        
print("\nMissing values after imputation:")
print(df.isnull().sum().sum())

**Explanation:** We identified columns with missing values and replaced them using the median of that column. We use the median because it is not heavily influenced by extreme values (outliers) unlike the mean.

## 3. Outlier Detection
Outliers are extreme values that deviate significantly from other observations. We detect them using the Interquartile Range (IQR) method.

In [ ]:
def detect_outliers(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Print number of outliers for continuous variables
for col in ['Annual_Rainfall', 'Seasonal_Rainfall']: # Focus on continuous meteorological data
    num_outliers, lb, ub = detect_outliers(df, col)
    print(f"{col}: {num_outliers} outliers detected")

**Explanation:** We use the IQR method to calculate lower and upper bounds. Any data point outside these boundaries is flagged as an outlier. We specifically focus on continuous numerical features like rainfall for outlier detection.

## 4. Outlier Treatment
Instead of removing outliers (which might lead to loss of valuable information), we will 'cap' or 'Winsorize' them. This means setting extreme values to a specified percentile boundary (e.g., 5th and 95th percentiles).

In [ ]:
def cap_outliers(data, col):
    lower_percentile = data[col].quantile(0.05)
    upper_percentile = data[col].quantile(0.95)
    data[col] = np.where(data[col] < lower_percentile, lower_percentile, data[col])
    data[col] = np.where(data[col] > upper_percentile, upper_percentile, data[col])
    return data

# Apply capping to continuous variables
for col in ['Annual_Rainfall', 'Seasonal_Rainfall']:
    df = cap_outliers(df, col)
    
print("Outliers capped successfully.")

**Explanation:** Capping ensures that extreme values are brought within a reasonable range (the 5th and 95th percentiles) so they don't skew the machine learning model's training process, while avoiding the deletion of rows.

## 5. Encoding Categorical Columns
Machine learning models require numerical input. We check if there are any categorical (text) columns that need encoding.

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Categorical columns found: {list(categorical_cols)}")

# If there were categorical columns, we would use LabelEncoder or OneHotEncoder here.
# Since the output is empty, our dataset is already completely numerical.
if len(categorical_cols) > 0:
    print("Encoding required.")
else:
    print("No categorical encoding required. All features are numerical.")

**Explanation:** Our dataset consists entirely of numerical values (either discrete scores or continuous measurements). Therefore, we don't need to perform techniques like One-Hot Encoding or Label Encoding.

## 6. Train Test Split
We divide the data into independent features (X) and the dependent target variable (y). Then we split them into a training set and a testing set.

In [ ]:
# Target is 'Flood'. We drop 'Flood_Probability' to prevent data leakage.
X = df.drop(['Flood', 'Flood_Probability'], axis=1) 
y = df['Flood']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

**Explanation:** We separated the features `X` from the target `y` (the binary `Flood` column). We also drop `Flood_Probability` to prevent data leakage, as it directly gives away the answer. We allocate 80% of the data to train the model and 20% to test its unseen accuracy.

## 7. Feature Scaling
Features with varying scales can negatively impact distance-based algorithms like KNN or gradient-based ones like XGBoost. We standardize our features to have a mean of 0 and a standard deviation of 1.

In [ ]:
scaler = StandardScaler()

# Fit on training data and transform both train and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for better readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Scaling complete. Example of scaled X_train:")
X_train_scaled.head()

**Explanation:** We use `StandardScaler` to normalize the range of independent variables. Crucially, we only `fit` the scaler on the training data, and then `transform` both the train and test data. This prevents 'data leakage' from the test set into the training process.

## 8. Save Preprocessing Objects
We save the trained scaler so it can be used later in our Flask web application to scale user inputs exactly like the training data.

In [ ]:
# Save the scaler object to the models folder
joblib.dump(scaler, '../models/scaler.save')

print("Scaler successfully saved to ../models/scaler.save")

**Explanation:** By saving the `StandardScaler` object as a file (`scaler.save`), we ensure that when our web app receives new data (e.g., rainfall input from a user), it can apply the exact same scaling transformation before making a prediction.